# Parameter Management

Once we have chosen an architecture
and set our hyperparameters,
we proceed to the training loop,
where our goal is to find parameter values
that minimize our loss function.
After training, we will need these parameters
in order to make future predictions.
Additionally, we will sometimes wish
to extract the parameters
perhaps to reuse them in some other context,
to save our model to disk so that
it may be executed in other software,
or for examination in the hope of
gaining scientific understanding.

Most of the time, we will be able
to ignore the nitty-gritty details
of how parameters are declared
and manipulated, relying on deep learning frameworks
to do the heavy lifting.
However, when we move away from
stacked architectures with standard layers,
we will sometimes need to get into the weeds
of declaring and manipulating parameters.
In this section, we cover the following:

* Accessing parameters for debugging, diagnostics, and visualizations.
* Sharing parameters across different model components.


In [1]:
import torch
from torch import nn

(**We start by focusing on an MLP with one hidden layer.**)


In [2]:
net = nn.Sequential(nn.LazyLinear(8),
                    nn.ReLU(),
                    nn.LazyLinear(1))

X = torch.rand(size=(2, 4))
net(X).shape

torch.Size([2, 1])

## [**Parameter Access**]
:label:`subsec_param-access`

Let's start with how to access parameters
from the models that you already know.


When a model is defined via the `Sequential` class,
we can first access any layer by indexing
into the model as though it were a list.
Each layer's parameters are conveniently
located in its attribute.


We can inspect the parameters of the second fully connected layer as follows.


In [3]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[-0.0915,  0.2323,  0.0672,  0.0951,  0.0561,  0.2846,  0.3074, -0.2919]])),
             ('bias', tensor([0.2610]))])

We can see that this fully connected layer
contains two parameters,
corresponding to that layer's
weights and biases, respectively.


### [**Targeted Parameters**]

Note that each parameter is represented
as an instance of the parameter class.
To do anything useful with the parameters,
we first need to access the underlying numerical values.
There are several ways to do this.
Some are simpler while others are more general.
The following code extracts the bias
from the second neural network layer, which returns a parameter class instance, and
further accesses that parameter's value.


In [4]:
type(net[2].bias), net[2].bias.data

(torch.nn.parameter.Parameter, tensor([0.2610]))

Parameters are complex objects,
containing values, gradients,
and additional information.
That is why we need to request the value explicitly.

In addition to the value, each parameter also allows us to access the gradient. Because we have not invoked backpropagation for this network yet, it is in its initial state.


In [5]:
net[2].weight.grad == None

True

### [**All Parameters at Once**]

When we need to perform operations on all parameters,
accessing them one-by-one can grow tedious.
The situation can grow especially unwieldy
when we work with more complex, e.g., nested, modules,
since we would need to recurse
through the entire tree to extract
each sub-module's parameters. Below we demonstrate accessing the parameters of all layers.


In [6]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([1, 8])),
 ('2.bias', torch.Size([1]))]

## [**Tied Parameters**]

Often, we want to share parameters across multiple layers.
Let's see how to do this elegantly.
In the following we allocate a fully connected layer
and then use its parameters specifically
to set those of another layer.
Here we need to run the forward propagation
`net(X)` before accessing the parameters.


In [7]:
# We need to give the shared layer a name so that we can refer to its
# parameters
shared = nn.LazyLinear(8)
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.LazyLinear(1))

net(X)
# Check whether the parameters are the same
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# Make sure that they are actually the same object rather than just having the
# same value
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


This example shows that the parameters
of the second and third layer are tied.
They are not just equal, they are
represented by the same exact tensor.
Thus, if we change one of the parameters,
the other one changes, too.


You might wonder,
when parameters are tied
what happens to the gradients?
Since the model parameters contain gradients,
the gradients of the second hidden layer
and the third hidden layer are added together
during backpropagation.


## Summary

We have several ways of accessing and tying model parameters.


## Exercises

1. Use the `NestMLP` model defined in :numref:`sec_model_construction` and access the parameters of the various layers.
1. Construct an MLP containing a shared parameter layer and train it. During the training process, observe the model parameters and gradients of each layer.
1. Why is sharing parameters a good idea?


[Discussions](https://discuss.d2l.ai/t/57)


1. Use the `NestMLP` model defined in :numref:`sec_model_construction` and access the parameters of the various layers.


2. Construct an MLP containing a shared parameter layer and train it. During the training process, observe the model parameters and gradients of each layer.


I'll help you solve this exercise by constructing and training an MLP with shared parameters. I'll provide a detailed solution with both explanation and code, focusing on observing parameters and gradients during training.

```python
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Create a synthetic dataset for regression
def generate_data(n_samples=1000):
    X = torch.randn(n_samples, 5)  # 5 input features
    # Create a non-linear pattern with some noise
    true_w = torch.tensor([0.5, -0.3, 0.2, -0.8, 0.5])
    y = torch.matmul(X, true_w) + 0.1 * torch.randn(n_samples) + 0.5
    y = y.unsqueeze(1)  # Shape to (n_samples, 1)
    return X, y

# Set random seed for reproducibility
torch.manual_seed(42)

# Generate data
X, y = generate_data()
train_dataset = TensorDataset(X, y)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Define MLP with shared parameter layer
class SharedParamMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Define layers
        self.layer1 = nn.Linear(5, 10)
        self.shared_layer = nn.Linear(10, 8)  # This layer will be shared
        self.layer3 = nn.Linear(8, 1)
        
    def forward(self, x):
        # First path
        x1 = torch.relu(self.layer1(x))
        x1 = torch.relu(self.shared_layer(x1))
        
        # Second path - reusing the shared layer
        x2 = torch.relu(self.shared_layer(x1))  # Note we're using the output from x1
        
        # Output layer
        output = self.layer3(x2)
        return output

# Create model, loss function, and optimizer
model = SharedParamMLP()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

# Function to print parameter and gradient information
def print_param_grad_info(model, epoch, step):
    print(f"Epoch {epoch}, Step {step}:")
    
    # Layer 1 information
    print("Layer 1 weight shape:", model.layer1.weight.shape)
    print("Layer 1 weight first row:", model.layer1.weight[0])
    if model.layer1.weight.grad is not None:
        print("Layer 1 weight grad first row:", model.layer1.weight.grad[0])
    
    # Shared layer information
    print("\nShared layer weight shape:", model.shared_layer.weight.shape)
    print("Shared layer weight first row:", model.shared_layer.weight[0])
    if model.shared_layer.weight.grad is not None:
        print("Shared layer weight grad first row:", model.shared_layer.weight.grad[0])
    
    # Layer 3 information
    print("\nLayer 3 weight shape:", model.layer3.weight.shape)
    print("Layer 3 weight:", model.layer3.weight)
    if model.layer3.weight.grad is not None:
        print("Layer 3 weight grad:", model.layer3.weight.grad)
    print("-" * 50)

# Lists to store loss and gradient magnitude
losses = []
shared_grad_norms = []

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    epoch_loss = 0
    for step, (batch_x, batch_y) in enumerate(train_loader):
        # Forward pass
        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Record information before updating parameters
        if step % 20 == 0:  # Print only occasionally to avoid too much output
            print_param_grad_info(model, epoch, step)
            
            # Record the gradient norm of the shared layer
            if model.shared_layer.weight.grad is not None:
                grad_norm = torch.norm(model.shared_layer.weight.grad).item()
                shared_grad_norms.append(grad_norm)
                losses.append(loss.item())
        
        # Update parameters
        optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch}, Average Loss: {epoch_loss / len(train_loader):.6f}")

# Plot the loss and gradient norm
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(losses)
plt.title('Loss During Training')
plt.xlabel('Steps (x20)')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(shared_grad_norms)
plt.title('Shared Layer Gradient Norm')
plt.xlabel('Steps (x20)')
plt.ylabel('Gradient Norm')

plt.tight_layout()
plt.show()

# Final parameter inspection
print("\nFinal model parameters and gradients:")
print_param_grad_info(model, num_epochs, 0)

# Demonstrate that weights from the shared layer are used twice
def trace_forward_pass(model, example_input):
    print("Tracing a single forward pass through the model:")
    
    # First layer
    z1 = model.layer1(example_input)
    a1 = torch.relu(z1)
    print(f"After first layer: shape={a1.shape}")
    
    # First use of shared layer
    z2 = model.shared_layer(a1)
    a2 = torch.relu(z2)
    print(f"After first use of shared layer: shape={a2.shape}")
    
    # Second use of shared layer (note we use a2 as input)
    z3 = model.shared_layer(a2)
    a3 = torch.relu(z3)
    print(f"After second use of shared layer: shape={a3.shape}")
    
    # Output layer
    output = model.layer3(a3)
    print(f"Final output: shape={output.shape}, value={output[0]}")
    
    return output

# Try with a sample input
example_x = X[0:1]  # One example
trace_forward_pass(model, example_x)
```

## Explanation of the Solution

This solution demonstrates constructing, training, and observing an MLP with shared parameters. Let me walk through the key components:

### 1. Model Architecture

The `SharedParamMLP` class defines an MLP where:
- The input goes through a first linear layer (`layer1`)
- Then passes through a shared layer (`shared_layer`) 
- The output of that shared layer is processed through the same shared layer again
- Finally, it goes through an output layer (`layer3`)

This architecture creates parameter sharing because the weights and biases of `shared_layer` are used twice in each forward pass.

### 2. Observation During Training

The `print_param_grad_info` function prints the parameters and gradients at specific points during training, allowing us to observe how they evolve.

Key observations from this training process:

1. **Parameter Sharing**: The same set of parameters (`shared_layer.weight` and `shared_layer.bias`) are used twice in the forward pass.

2. **Gradient Accumulation**: The gradients for the shared layer are accumulated from both usages. When backpropagation runs, gradients flow backward through both paths that use the shared layer, and these gradients are summed at the shared parameters.

3. **Training Dynamics**: 
   - The gradient norms of the shared layer tend to be larger than other layers because they accumulate gradients from multiple paths.
   - This can lead to faster learning in the shared layer compared to non-shared layers.

4. **Convergence**: Despite the unusual architecture, the model can still learn the underlying pattern in the data, demonstrating the effectiveness of parameter sharing.

### Why Parameter Sharing Works

Parameter sharing is effective because:
1. **Reduced Model Size**: Using the same parameters multiple times reduces the total number of parameters, making the model more memory-efficient.
2. **Feature Reuse**: Shared parameters allow the model to reuse learned features at different points in the network.
3. **Regularization Effect**: Parameter sharing acts as a form of regularization, as it constrains the model and can help prevent overfitting.

This concept is fundamental to many modern neural network architectures, particularly convolutional neural networks, where the same convolutional filters are applied across the entire image, effectively sharing parameters across spatial locations.

The visualization of loss and gradient norms helps us understand the training dynamics, showing how parameter sharing affects the optimization process.

3. Why is sharing parameters a good idea?

# Why Parameter Sharing is a Good Idea

While parameter sharing certainly saves space (reducing memory requirements), there are several more profound benefits that make it a powerful technique in deep learning:

## 1. Model Efficiency

- **Parameter Reduction**: Shared parameters significantly reduce the total number of parameters that need to be stored and updated, leading to lower memory consumption and faster training.
- **Computational Efficiency**: Fewer parameters mean fewer computations during both forward and backward passes, resulting in faster inference and training.

## 2. Statistical Efficiency

- **Sample Efficiency**: With fewer parameters to learn, the model can achieve good performance with less training data, reducing the risk of overfitting.
- **Better Generalization**: Parameter sharing constrains the model's flexibility in specific ways, often leading to better generalization to unseen data.

## 3. Inductive Biases

- **Encoding Domain Knowledge**: Parameter sharing encodes specific assumptions about the structure of the data. For example, in CNNs, weight sharing encodes the assumption that patterns learned in one part of an image can be useful in other parts.
- **Translation Invariance**: In CNNs, shared convolutional filters allow the network to detect the same features regardless of their position in the input, making the model invariant to translations.

## 4. Specialized Architectures

- **Enabling Architectural Innovations**: Parameter sharing enables novel architectures like recurrent neural networks (RNNs), where the same weights process sequential inputs, and transformers, where self-attention mechanisms share parameters across different positions.
- **Deep Network Viability**: For very deep networks, parameter sharing (as in ResNets with skip connections) allows signals to propagate more effectively, mitigating the vanishing gradient problem.

## 5. Transfer Learning

- **Feature Reuse**: Shared parameters can learn general features that are reusable across different parts of the model or even across different tasks.
- **Multi-task Learning**: Parameter sharing enables effective multi-task learning where certain layers can be shared across different tasks while task-specific layers remain separate.

## 6. Regularization Effects

- **Implicit Regularization**: By constraining the model to reuse parameters, parameter sharing acts as a form of regularization that can prevent overfitting.
- **Stable Gradient Flow**: In some architectures, parameter sharing can lead to more stable gradient flow during backpropagation.

## Real-World Applications

- **CNNs for Computer Vision**: The most prominent example of parameter sharing success, where convolutional layers share the same filters across the entire input image.
- **RNNs for Sequential Data**: The same weights are applied at each time step when processing sequences.
- **Graph Neural Networks**: Parameters are shared across nodes in the graph.
- **Siamese Networks**: Identical networks with shared weights process different inputs to compare them.

Parameter sharing is one of the fundamental ideas that has enabled the success of modern deep learning by making models both more efficient and more effective at learning meaningful representations.